# Performance - algoritmo ineficiente

Notebook para registrar metricas emitidas pelo firmware em `PERF_DATA,...` e salvar os resultados em `performance_ineficiente.csv`.

In [ ]:
from __future__ import annotations

import csv
from datetime import datetime, timezone
from pathlib import Path

CSV_PATH = Path("performance_ineficiente.csv")
COLUMNS = [
    "run_id",
    "timestamp",
    "samples",
    "insert_us_total",
    "insert_us_avg",
    "publish_us_total",
    "publish_us_avg",
    "free_heap",
    "min_free_heap",
    "publish_failures",
]

def load_rows(path: Path = CSV_PATH) -> list[dict[str, str]]:
    if not path.exists():
        return []
    with path.open(newline="", encoding="utf-8") as csv_file:
        return list(csv.DictReader(csv_file))

def save_rows(rows: list[dict[str, object]], path: Path = CSV_PATH) -> None:
    with path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=COLUMNS)
        writer.writeheader()
        for row in rows:
            writer.writerow({column: row.get(column, "") for column in COLUMNS})

rows = load_rows()
print(f"{len(rows)} registros carregados de {CSV_PATH}")

Cole abaixo uma linha do monitor serial no formato:

`PERF_DATA,1,30015,42,1234,29.38,90000,2142.86,180000,170000,0`

In [ ]:
def parse_perf_data(line: str) -> dict[str, object]:
    parts = [part.strip() for part in line.strip().split(",")]
    if len(parts) != 11 or parts[0] != "PERF_DATA":
        raise ValueError("Linha esperada: PERF_DATA,run_id,timestamp_ms,samples,insert_us_total,insert_us_avg,publish_us_total,publish_us_avg,free_heap,min_free_heap,publish_failures")

    return {
        "run_id": int(parts[1]),
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "samples": int(parts[3]),
        "insert_us_total": int(parts[4]),
        "insert_us_avg": float(parts[5]),
        "publish_us_total": int(parts[6]),
        "publish_us_avg": float(parts[7]),
        "free_heap": int(parts[8]),
        "min_free_heap": int(parts[9]),
        "publish_failures": int(parts[10]),
    }

def append_perf_line(line: str, path: Path = CSV_PATH) -> dict[str, object]:
    rows = load_rows(path)
    row = parse_perf_data(line)
    rows.append(row)
    save_rows(rows, path)
    return row

# Exemplo de uso:
# append_perf_line("PERF_DATA,1,30015,42,1234,29.38,90000,2142.86,180000,170000,0")

In [ ]:
# Registro manual alternativo, caso voce queira digitar os valores sem colar a linha serial.
def append_manual_row(
    run_id: int,
    samples: int,
    insert_us_total: int,
    publish_us_total: int,
    free_heap: int,
    min_free_heap: int,
    publish_failures: int = 0,
    path: Path = CSV_PATH,
) -> dict[str, object]:
    row = {
        "run_id": run_id,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "samples": samples,
        "insert_us_total": insert_us_total,
        "insert_us_avg": insert_us_total / samples if samples else 0.0,
        "publish_us_total": publish_us_total,
        "publish_us_avg": publish_us_total / samples if samples else 0.0,
        "free_heap": free_heap,
        "min_free_heap": min_free_heap,
        "publish_failures": publish_failures,
    }
    rows = load_rows(path)
    rows.append(row)
    save_rows(rows, path)
    return row

# Exemplo de uso:
# append_manual_row(run_id=1, samples=100, insert_us_total=3000, publish_us_total=250000, free_heap=180000, min_free_heap=170000)

In [ ]:
rows = load_rows()
rows[-5:] if rows else []